## Feature Processing

In [26]:
# Import libraries 
import pandas as pd 
import numpy as np 
import os, sys 

sys.path.append(os.path.abspath(".."))

### Approach

+ Data Preprocessing: preprocessing on dataset Information data & Visual data
+ Data Transformation: apply encoding on binary features (hardware data)
+ Data Storage

## Data Loading

In [27]:
# Implement functionalities
from scripts.collection.collector import DataLoader

In [28]:
# Load 4 datasets 
data_loader = DataLoader(folder="processed")
visual_data = data_loader.load("visual_table.csv")
information_data = data_loader.load("memory_table.csv")

File accepted
File accepted


In [29]:
# Visual data
visual_data

,SCREEN_ID,SCREEN_SIZE_INCH,COLOR,WIDTH_DISPLAY,HEIGHT_DISPLAY,DISPLAY,WEBCAM(BUILT-IN)
0,S0,14.00,gray,2160.0,1440.0,NO,YES
1,S1,14.00,black,1920.0,1080.0,YES,YES
2,S2,11.60,black,1366.0,768.0,YES,YES
3,S3,12.50,other,1366.0,768.0,YES,YES
4,S4,11.60,black,1366.0,768.0,NO,YES
...,...,...,...,...,...,...,...
4178,S4178,8.43,other,1920.0,1080.0,NO,NO
4179,S4179,12.50,black,1920.0,1080.0,NO,NO
4180,S4180,8.43,other,1920.0,1080.0,NO,YES
4181,S4181,8.43,black,1920.0,1080.0,NO,NO


In [30]:
# Information data
information_data

,MEMORY_ID,HARD_DRIVE,HARD_DRIVE_CAPACITY_UNIT,SSD_CAPACITY,SSD_CAPACITY_UNIT
0,M0,512,gb,1,tb
1,M1,500,gb,500,gb
2,M2,16,gb,240,unknown
3,M3,256,unknown,256,gb
4,M4,256,unknown,16,gb
...,...,...,...,...,...
4177,M4177,256,unknown,240,unknown
4178,M4178,256,unknown,240,unknown
4179,M4179,256,unknown,120,gb
4180,M4180,256,unknown,240,unknown


## Data Preprocessing: Information data

**Feature 1: Hard Drive Capacity**

In [31]:
dataset1 = information_data.copy()
hard_drive_data = dataset1[["HARD_DRIVE", "HARD_DRIVE_CAPACITY_UNIT"]]
hard_drive_data.loc[:, "HARD_DRIVE"] = hard_drive_data[["HARD_DRIVE", "HARD_DRIVE_CAPACITY_UNIT"]].apply(lambda row: 
                                                                                                         row["HARD_DRIVE"]*1000 if row["HARD_DRIVE_CAPACITY_UNIT"] == "tb" else row["HARD_DRIVE"], axis=1)

hard_drive_data.loc[:, "HARD_DRIVE_CAPACITY_UNIT"] = hard_drive_data[["HARD_DRIVE", "HARD_DRIVE_CAPACITY_UNIT"]].apply(lambda row:  
                                                                                                                       "gb" if row["HARD_DRIVE_CAPACITY_UNIT"] == "tb" else row["HARD_DRIVE_CAPACITY_UNIT"], axis=1)
hard_drive_data[hard_drive_data["HARD_DRIVE_CAPACITY_UNIT"].str.contains("unknown")].value_counts()

HARD_DRIVE  HARD_DRIVE_CAPACITY_UNIT
256         unknown                     3078
Name: count, dtype: int64

In [32]:
# Show dataset
hard_drive_data[hard_drive_data["HARD_DRIVE"] == 256]
hard_drive_data.loc[:, "HARD_DRIVE_CAPACITY_UNIT"] = hard_drive_data["HARD_DRIVE_CAPACITY_UNIT"].str.replace("unknown", "gb")
hard_drive_data

# hard_drive_data[hard_drive_data["HARD_DRIVE_CAPACITY_UNIT"] == "unknown"]["HARD_DRIVE"].value_counts()

,HARD_DRIVE,HARD_DRIVE_CAPACITY_UNIT
0,512,gb
1,500,gb
2,16,gb
3,256,gb
4,256,gb
...,...,...
4177,256,gb
4178,256,gb
4179,256,gb
4180,256,gb


**Feature 2: SSD Capacity**

In [33]:
# Extract feature SSD capacity
ssd_capacity_data = information_data[["SSD_CAPACITY","SSD_CAPACITY_UNIT"]]
ssd_capacity_data

# Process data: converting non-gb values to gb values
ssd_capacity_data.loc[:, "SSD_CAPACITY"] = ssd_capacity_data[["SSD_CAPACITY", "SSD_CAPACITY_UNIT"]].apply(lambda row: 
                                                                                                         row["SSD_CAPACITY"]*1000 if row["SSD_CAPACITY_UNIT"] == "tb" else row["SSD_CAPACITY"], axis=1)

ssd_capacity_data.loc[:, "SSD_CAPACITY_UNIT"] = ssd_capacity_data[["SSD_CAPACITY", "SSD_CAPACITY_UNIT"]].apply(lambda row:  
                                                                                                                       "gb" if row["SSD_CAPACITY_UNIT"] == "tb" else row["SSD_CAPACITY_UNIT"], axis=1)
ssd_capacity_data[ssd_capacity_data["SSD_CAPACITY_UNIT"].str.contains("unknown")].value_counts()

SSD_CAPACITY  SSD_CAPACITY_UNIT
240           unknown              2053
Name: count, dtype: int64

In [34]:
# Remove unknown value in ssd capacity unit feature
ssd_capacity_data.loc[:, "SSD_CAPACITY_UNIT"] = ssd_capacity_data["SSD_CAPACITY_UNIT"].str.replace("unknown", "gb")
ssd_capacity_data

,SSD_CAPACITY,SSD_CAPACITY_UNIT
0,1000,gb
1,500,gb
2,240,gb
3,256,gb
4,16,gb
...,...,...
4177,240,gb
4178,240,gb
4179,120,gb
4180,240,gb


## Data Transformation

In [35]:
# Import functionalities for transforming data
from scripts.processing.transformer import ColumnTransformer

In [39]:
# Original information dataset
information_data
information_data1 = information_data.drop(columns=["HARD_DRIVE", "HARD_DRIVE_CAPACITY_UNIT", "SSD_CAPACITY", "SSD_CAPACITY_UNIT"], axis=1)
information_data1

,MEMORY_ID
0,M0
1,M1
2,M2
3,M3
4,M4
...,...
4177,M4177
4178,M4178
4179,M4179
4180,M4180


In [40]:
# Merge harddrive and ssd capacity
memory_data = pd.concat([hard_drive_data, ssd_capacity_data], axis=1)
memory_data

,HARD_DRIVE,HARD_DRIVE_CAPACITY_UNIT,SSD_CAPACITY,SSD_CAPACITY_UNIT
0,512,gb,1000,gb
1,500,gb,500,gb
2,16,gb,240,gb
3,256,gb,256,gb
4,256,gb,16,gb
...,...,...,...,...
4177,256,gb,240,gb
4178,256,gb,240,gb
4179,256,gb,120,gb
4180,256,gb,240,gb


In [42]:
# Identify container object: HARD_DRIVE_CAPACITY, SSD_CAPACITY
selected_features = ["HARD_DRIVE","SSD_CAPACITY"]
memory_container = [information_data1, memory_data[["HARD_DRIVE", "SSD_CAPACITY"]]]

# Combine dataset 
column_transformer = ColumnTransformer()
dataset = column_transformer.combine(memory_container)
dataset

Number of datasets for Column Combination: 2


,MEMORY_ID,HARD_DRIVE,SSD_CAPACITY
0,M0,512,1000
1,M1,500,500
2,M2,16,240
3,M3,256,256
4,M4,256,16
...,...,...,...
4177,M4177,256,240
4178,M4178,256,240
4179,M4179,256,120
4180,M4180,256,240


## Data Storage

In [44]:
# Implement functionalities
from scripts.collection.saver import OneFileSaver

In [46]:
# Save the data
datasaver = OneFileSaver(folder="data/processed/v2")
datasaver.save(dataset, "information_data_v2")

File information_data_v2.csv has been stored successfully


## Conclusion

Dataset description:
+ name: information_data_v2.csv
+ features: MEMORY_ID, HARD_DRIVE, SSD_CAPACITY
+ number_features: 3


Created by Adoan Mian at 17/03/2026
